In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from models.CLPTransformer_v4 import CLPTransformer
from training import load_model

model_name = "CLPTransformer_v4_CL"

model = load_model(CLPTransformer, model_name)

In [ ]:
from solvers.bsm_vcs import BSM_VCS_Solver

instance_file = "benchmarks/BR4.txt"
instance_number = 1
w = 8

bsm_solver = BSM_VCS_Solver(model, w)
bsm_eval = bsm_solver.solve(instance_file, instance_number)
print(bsm_eval)

TypeError: BSM_VCS_Solver.solve() takes 3 positional arguments but 4 were given

In [2]:
from solvers.bsm_gm import BSM_GM_Solver

instance_file = "benchmarks/BR4.txt"
instance_number = 1
w = 2

bsm_solver = BSM_GM_Solver(model)
bsm_eval = bsm_solver.solve(instance_file, instance_number, w)
print(bsm_eval)

94.18355898146936


In [10]:
from solvers.bsg import BSGSolver

instance_file = "benchmarks/BR4.txt"
instance_number = 1
w = 8

bsg_solver = BSGSolver()
bsg_eval = bsg_solver.solve(instance_file, instance_number, w)
print(bsg_eval)

95.51205


In [ ]:
from solvers.bsm_gm import BSMSolver

instance_file = "benchmarks/BR4.txt"
w = 2

evals = []
for instance_number in range(100):
    bsm_solver = BSMSolver(model)
    bsm_eval = bsm_solver.solve(instance_file, instance_number, w)
    evals.append(bsm_eval)
    
print(evals)
print(sum(evals) / len(evals))

KeyboardInterrupt: 

In [4]:
from solvers.bsg import BSGSolver

instance_file = "benchmarks/BR4.txt"
w = 8

evals = []
for instance_number in range(100):
    bsg_solver = BSGSolver()
    bsg_eval = bsg_solver.solve(instance_file, instance_number, w)
    evals.append(bsg_eval)
    
print(evals)
print(sum(evals) / len(evals))

[95.245118, 95.51205, 94.883252, 95.986151, 95.748378, 95.215141, 95.464509, 93.013916, 95.514988, 95.310469, 94.692904, 94.208268, 95.601962, 94.465866, 95.415648, 95.142923, 95.275314, 93.123124, 96.075188, 96.709673, 95.422714, 94.526843, 94.939039, 94.593647, 95.479062, 95.340061, 94.152508, 96.274722, 95.252383, 95.937699, 94.948657, 95.526358, 96.090865, 94.76109, 95.140603, 95.98652, 96.114195, 94.63952, 96.009418, 93.850973, 94.470385, 95.665562, 94.59264, 93.335649, 95.692079, 94.589696, 93.682938, 95.896914, 94.16188, 95.866182, 96.09366, 95.393109, 94.405566, 95.562124, 95.623421, 96.57761, 95.133664, 94.443532, 95.746121, 95.095355, 95.997683, 96.343905, 95.647556, 94.417071, 96.964654, 96.207001, 95.179082, 94.987873, 95.474828, 94.698361, 95.591975, 94.434443, 93.999213, 93.027868, 95.018947, 95.056169, 96.141151, 94.711764, 97.18382, 94.509661, 95.48643, 95.530641, 96.199118, 93.831713, 95.762223, 96.075577, 94.522686, 95.62828, 94.224241, 94.65967, 95.490448, 94.619769,

In [6]:
from solvers.bsg import BSGSolver

instance_file = "benchmarks/BR4.txt"
instance_number = 1
w = 16

bsg_solver = BSGSolver()
bsg_eval = bsg_solver.solve(instance_file, instance_number, w)
print(bsg_eval)

96.497104


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from solvers.sequential.greedy import GreedyModelSolver
from solvers.vcs import VCSSolver

instance_file = "benchmarks/BR7.txt"
w = 8

def run_instance(instance_number):
    model_solver = GreedyModelSolver(model)
    vcs_solver = VCSSolver()

    model_eval = model_solver.solve(instance_file, instance_number, w)
    vcs_eval = vcs_solver.solve(instance_file, instance_number)

    return instance_number, {'Model': model_eval, 'VCS': vcs_eval}


if __name__ == "__main__":
    results = {}

    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = [executor.submit(run_instance, i) for i in range(10)]

        for future in as_completed(futures):
            instance_number, result = future.result()
            results[instance_number] = result

    print(results)

{2: {'Model': 89.5624, 'VCS': 91.002146}, 4: {'Model': 92.548, 'VCS': 92.35321}, 7: {'Model': 89.34609999999999, 'VCS': 90.314361}, 0: {'Model': 93.76400000000001, 'VCS': 93.197415}, 1: {'Model': 90.4269, 'VCS': 91.89574}, 8: {'Model': 91.79469999999999, 'VCS': 91.370659}, 9: {'Model': 92.95219999999999, 'VCS': 90.343088}, 5: {'Model': 91.3886, 'VCS': 92.497685}, 6: {'Model': 92.0765, 'VCS': 91.984535}, 3: {'Model': 91.6176, 'VCS': 93.381635}}


In [ ]:
import pandas as pd
df = pd.DataFrame(results).T.sort_index()
df['diff'] = df['Model'] - df['VCS']
print(df.mean())

Model    91.547700
VCS      91.834047
diff     -0.286347
dtype: float64


In [ ]:
from concurrent.futures import ProcessPoolExecutor
from solvers.bsg import BSGSolver
from solvers.bsm_gm import BSMSolver

instance_file = "benchmarks/BR4.txt"
num_instances = 20
w = 8

def run_bsg(instance_number):
    solver = BSGSolver()
    return solver.solve(instance_file, instance_number, w)

def run_bsm(instance_number):
    solver = BSMSolver(model)
    return solver.solve(instance_file, instance_number, w)

if __name__ == '__main__':
    results = {}
    for i in range(num_instances):
        print("Resolviendo instancia", i+1)
        with ProcessPoolExecutor(max_workers=2) as executor:
            future_bsg = executor.submit(run_bsg, instance_number=i)
            future_bsm = executor.submit(run_bsm, instance_number=i)

            bsg_eval = future_bsg.result()
            bsm_eval = future_bsm.result()
            
            results[i] = {
                'BSG': bsg_eval,
                'BSM': bsm_eval
            }

Resolviendo instancia 1
['0.924979', '0.871469', '0.926189', '0.933577', '0.906179', '0.883201', '0.884223', '0.884389', '8']
['0.871469', '0.900795', '0.900795', '0.914847', '0.889195', '0.894189', '0.899556', '0.899556', '0.929441', '0.929441', '0.883201', '0.91488', '0.903869', '0.892479', '0.891385', '0.873011', '0.884223', '0.899541', '0.885271', '0.908338', '0.892572', '0.885935', '0.891385', '0.909822', '0.903768', '0.884389', '0.924979', '0.903768', '0.923143', '0.873606', '0.907948', '0.900426', '0.906179', '0.89906', '0.891941', '0.887401', '0.911951', '0.895584', '0.89833', '0.893305', '0.924979', '0.871469', '0.870298', '0.895584', '0.920129', '0.916634', '0.911875', '0.91694', '0.891941', '0.908085', '0.926189', '0.908021', '0.89026', '0.930625', '0.901164', '0.903768', '0.933577', '0.901967', '0.865357', '0.891373', '0.909475', '0.89833', '0.933637', '0.918783', '8']
['0.895584', '0.892981', '0.883129', '0.903768', '0.865357', '0.871763', '0.903768', '0.869634', '0.870298

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T.sort_index()
df['diff'] = df['BSM'] - df['BSG']
print(df.mean())
df.tail(50)

BSG     95.130242
BSM     93.917175
diff    -1.213067
dtype: float64


,BSG,BSM,diff
0,95.245118,93.3637,-1.881418
1,95.512050,94.3569,-1.155150
2,94.883252,93.7176,-1.165652
3,95.986151,94.6431,-1.343051
4,95.748378,94.7641,-0.984278
5,95.215141,94.2830,-0.932141
6,95.464509,94.1374,-1.327109
7,93.013916,91.2666,-1.747316
8,95.514988,94.7826,-0.732388
9,95.310469,95.2260,-0.084469


In [ ]:
from concurrent.futures import ProcessPoolExecutor
from solvers.timed.timed_bsg import TimedBSGSolver
from solvers.timed.timed_bsm import TimedBSMSolver

instance_file = "benchmarks/BR4.txt"
instance_number = 1
time = 30

def run_bsg():
    solver = TimedBSGSolver()
    return solver.solve(instance_file, instance_number, time)

def run_bsm():
    solver = TimedBSMSolver(model)
    return solver.solve(instance_file, instance_number, time, verbose=True)

if __name__ == '__main__':
    with ProcessPoolExecutor(max_workers=2) as executor:
        future_bsg = executor.submit(run_bsg)
        future_bsm = executor.submit(run_bsm)

        timed_bsg_eval = future_bsg.result()
        timed_bsm_eval = future_bsm.result()

    print("Timed BSG:", timed_bsg_eval)
    print("Timed BSM:", timed_bsm_eval)

Ejecutando BSM con w = 1
Actualizando mejor volumen: 92.5136
Ejecutando BSM con w = 2
Actualizando mejor volumen: 93.3528
Ejecutando BSM con w = 3
Ejecutando BSM con w = 5
Ejecutando BSM con w = 8
Actualizando mejor volumen: 95.0251
Ejecutando BSM con w = 12
Ejecutando BSM con w = 17
Ejecutando BSM con w = 25
Ejecutando BSM con w = 36
Timed BSG: 96.497104
Timed BSM: 95.0251


In [ ]:
from concurrent.futures import ProcessPoolExecutor
from solvers.timed.timed_bsg import TimedBSGSolver
from solvers.timed.timed_bsm import TimedBSMSolver

instance_file = "benchmarks/BR4.txt"
num_instances = 20
time = 30

def run_bsg(instance_number):
    solver = TimedBSGSolver()
    return solver.solve(instance_file, instance_number, time)

def run_bsm(instance_number):
    solver = TimedBSMSolver(model)
    return solver.solve(instance_file, instance_number, time)

if __name__ == '__main__':
    results = {}
    for i in range(num_instances):
        print("Resolviendo instancia", i+1)
        with ProcessPoolExecutor(max_workers=2) as executor:
            future_bsg = executor.submit(run_bsg, instance_number=i)
            future_bsm = executor.submit(run_bsm, instance_number=i)

            timed_bsg_eval = future_bsg.result()
            timed_bsm_eval = future_bsm.result()
            
            results[i] = {
                'Timed BSG': timed_bsg_eval,
                'Timed BSM': timed_bsm_eval
            }

Resolviendo instancia 0
Resolviendo instancia 1
Resolviendo instancia 2
Resolviendo instancia 3
Resolviendo instancia 4
Resolviendo instancia 5
Resolviendo instancia 6
Resolviendo instancia 7
Resolviendo instancia 8
Resolviendo instancia 9
Resolviendo instancia 10
Resolviendo instancia 11
Resolviendo instancia 12
Resolviendo instancia 13
Resolviendo instancia 14
Resolviendo instancia 15
Resolviendo instancia 16
Resolviendo instancia 17
Resolviendo instancia 18
Resolviendo instancia 19


In [ ]:
import pandas as pd
df = pd.DataFrame(results).T.sort_index()
df['diff'] = df['Timed BSM'] - df['Timed BSG']
print(df.mean())

Timed BSG    95.901950
Timed BSM    94.602595
diff         -1.299355
dtype: float64


In [ ]:
df

,Timed BSG,Timed BSM,diff
0,95.245118,93.3577,-1.887418
1,96.497104,95.2011,-1.296004
2,95.891085,94.1940,-1.697085
3,96.287849,95.2073,-1.080549
4,96.240155,94.6353,-1.604855
5,95.960560,94.8463,-1.114260
6,96.195632,94.6429,-1.552732
7,94.634778,93.6119,-1.022878
8,96.351250,94.8163,-1.534950
9,95.832716,95.2260,-0.606716


In [ ]:
from concurrent.futures import ProcessPoolExecutor
from solvers.bsg import BSGSolver
from solvers.bsm_gm import BSMSolver

instance_file = "benchmarks/BR4.txt"
instance_number = 0
w = 42

solver = BSGSolver()
bsg_sol = solver.solve(instance_file, instance_number, w)
print(bsg_sol)

95.219056


In [ ]:
from importlib import reload
import solvers.bsm_gm
reload(solvers.bsm_gm)

<module 'solvers.bsm' from '/home/oscar/CLP-Framework/src/solvers/bsm.py'>

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from solvers.bsg import BSGSolver
from solvers.bsm_gm import BSMSolver

instance_file = "benchmarks/BR4.txt"
instance_number = 0
w = 42

solver = BSMSolver(model)
bsm_sol = solver.solve(instance_file, instance_number, w)
print(bsm_sol)

1
37
42
42
42
42
42
42
18
9
317
92.9588
